In [1]:
import os

os.environ["HF_HOME"] = os.environ["SCRATCH"] + "/.cache"
os.environ["HF_TOKEN"] = (
    open(os.path.expanduser("~/.cache/huggingface/token")).read().strip()
)
os.environ["UV_CACHE_DIR"] = os.environ["SCRATCH"] + "/.cache"
NUM_CPUS = 8

In [2]:
from datasets import load_dataset

lmsys = load_dataset("lmsys/lmsys-chat-1m")
wildchat = load_dataset("allenai/WildChat")

data/train-00000-of-00006-4feeb3f83346a0(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00006-4030672591c2f4(…):   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00002-of-00006-1779b7cec94621(…):   0%|          | 0.00/250M [00:00<?, ?B/s]

data/train-00003-of-00006-2fa862bfed56af(…):   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00004-of-00006-18f4bdd50c103e(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

...

data/train-00002-of-00006.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

data/train-00003-of-00006.parquet:   0%|          | 0.00/269M [00:00<?, ?B/s]

data/train-00004-of-00006.parquet:   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00005-of-00006.parquet:   0%|          | 0.00/231M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/529428 [00:00<?, ? examples/s]

### 1) Sanitizing LMSYS/WildChat with OpenAI moderation labels

In [ ]:
def verify_moderation(entry: dict):
    for entity in entry["openai_moderation"]:
        for key in entity["categories"].keys():
            if entity["categories"][key]:
                return False
    return True


def wildchat_clean_conversation(sample: dict) -> dict:
    """Remove useless conversation features"""
    conversation = sample["conversation"]
    for i, entry in enumerate(conversation):
        conversation[i] = {"content": entry["content"], "role": entry["role"]}
    del sample["conversation"]
    sample["conversation"] = conversation
    return sample

In [4]:
from datasets import Features, Value

detox_lmsys = lmsys["train"].filter(lambda x: verify_moderation(x), num_proc=NUM_CPUS)
detox_wildchat = wildchat["train"].filter(
    lambda x: verify_moderation(x), num_proc=NUM_CPUS
)

target_features = detox_wildchat.features.copy()
target_features["conversation"] = [
    {"content": Value("string"), "role": Value("string")}
]

detox_wildchat = detox_wildchat.map(
    lambda x: wildchat_clean_conversation(x),
    num_proc=NUM_CPUS,
    features=Features(target_features),
)

Filter (num_proc=8):   0%|          | 0/1000000 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/529428 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/524274 [00:00<?, ? examples/s]

In [5]:
wildchat_clean_conversation(detox_wildchat[0])["conversation"]

[{'content': 'Write a very long, elaborate, descriptive and detailed shooting script, including a background and dialogues, for a John Waters-inspired film scene that includes a character preparing omelettes. You are free to choose the setting, scenario (it should make sense) and characters (give them names, and describe their appearance and clothing in detail).',
  'role': 'user'},
 {'content': 'Title: "Divine Decadence: Omelette a la Waters"\n\nEXT. MORNINGSIDE EDGE TRAILER PARK - DAY\n\nThe sun casts a warm glow over the eclectic collection of trailers that make up the Morningside Edge Trailer Park. In the heart of this kitschy, suburban vortex, we find the largest and most garish trailer of them all: THE PINK PALACE, home to our film\'s protagonists.\n\nINT. PINK PALACE - KITCHEN - DAY\n\nThe Pink Palace\'s kitchen is a true wonder to behold. Every available inch of counter space is covered in campy knick-knacks, ironic food-themed art and vintage appliances. The wallpaper features

In [6]:
detox_wildchat[0]["conversation"]

[{'content': 'Write a very long, elaborate, descriptive and detailed shooting script, including a background and dialogues, for a John Waters-inspired film scene that includes a character preparing omelettes. You are free to choose the setting, scenario (it should make sense) and characters (give them names, and describe their appearance and clothing in detail).',
  'role': 'user'},
 {'content': 'Title: "Divine Decadence: Omelette a la Waters"\n\nEXT. MORNINGSIDE EDGE TRAILER PARK - DAY\n\nThe sun casts a warm glow over the eclectic collection of trailers that make up the Morningside Edge Trailer Park. In the heart of this kitschy, suburban vortex, we find the largest and most garish trailer of them all: THE PINK PALACE, home to our film\'s protagonists.\n\nINT. PINK PALACE - KITCHEN - DAY\n\nThe Pink Palace\'s kitchen is a true wonder to behold. Every available inch of counter space is covered in campy knick-knacks, ironic food-themed art and vintage appliances. The wallpaper features

In [ ]:
{
    "Retained LMSYS": round(len(detox_lmsys) / len(lmsys["train"]), 2),
    "Retained WildChat": round(len(detox_wildchat) / len(wildchat["train"]), 2),
}

#### Analyzing lengths

In [8]:
import torch
from transformers import AutoTokenizer

llama_tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")

In [9]:
def sample_n_tokens(entry: dict, tokenizer: object) -> dict:
    tokenized = tokenizer.apply_chat_template(
        entry["conversation"], add_generation_prompt=True, return_tensors="pt"
    )
    n_tokens = tokenized["input_ids"].shape[-1]
    entry["n_tokens"] = n_tokens
    return entry


def get_n_tokens(dataset: object, tokenizer: object) -> int:
    """Main function to get dataset tokens with a given tokenizer"""
    data_tokens = dataset.map(
        lambda x: sample_n_tokens(x, tokenizer), num_proc=NUM_CPUS
    )
    num_tokens = torch.tensor(data_tokens["n_tokens"]).sum().item()
    return num_tokens

In [10]:
# lmsys_n_tokens = get_n_tokens(dataset=detox_lmsys, tokenizer=llama_tok)
# wildchat_n_tokens = get_n_tokens(dataset=detox_wildchat, tokenizer=llama_tok)

In [11]:
# print(f"#Llama Tokens LMSYS: {lmsys_n_tokens}")
# print(f"#Llama Tokens WildChat: {wildchat_n_tokens}")

### 2) Sanitizing WildJailbreak and WildGuardMix

#### Wildguard

In [12]:
wildguard = load_dataset("allenai/wildguardmix", "wildguardtrain")

train/wildguard_train.parquet:   0%|          | 0.00/53.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/86759 [00:00<?, ? examples/s]

In [ ]:
# Wildguard
def sample_sanitize_wildguard(sample: dict) -> dict:
    return (
        not sample["adversarial"]
        and sample["prompt_harm_label"] != "harmful"
        and sample["response_harm_label"] != "harmful"
    )


detox_wildguard = wildguard["train"].filter(lambda x: sample_sanitize_wildguard(x))
round(len(detox_wildguard) / len(wildguard["train"]), 2)

#### Wildjailbreak

In [14]:
wildjailbreak = load_dataset(
    "allenai/wildjailbreak", "train", delimiter="\t", keep_default_na=False
)

In [ ]:
# Wildchat
def sample_sanitize_wildjb(sample: dict) -> dict:
    return "harmful" not in sample["data_type"]


detox_wildjb = wildjailbreak["train"].filter(lambda x: sample_sanitize_wildjb(x))
round(len(detox_wildjb) / len(wildjailbreak["train"]), 2)

### Putting all together

In [ ]:
def sample_format_conversation_wildjb(sample: dict) -> dict:
    if sample["adversarial"] is None:
        txt_field = "vanilla"
    else:
        txt_field = "adversarial"
    conversation = [
        {"content": sample[txt_field], "role": "user"},
        {"content": sample["completion"], "role": "assistant"},
    ]
    sample["conversation"] = conversation
    return sample


def sample_format_conversation_wildguard(sample: dict) -> dict:
    conversation = [
        {"content": sample["prompt"], "role": "user"},
        {"content": sample["response"], "role": "assistant"},
    ]
    sample["conversation"] = conversation
    return sample

In [17]:
detox_wildjb = detox_wildjb.map(lambda x: sample_format_conversation_wildjb(x))
detox_wildguard = detox_wildguard.map(lambda x: sample_format_conversation_wildguard(x))

Map:   0%|          | 0/128781 [00:00<?, ? examples/s]

Map:   0%|          | 0/20137 [00:00<?, ? examples/s]

In [ ]:
def label_dataset_sample(sample: dict, label: str) -> dict:
    sample["origin"] = label
    return sample


def remove_useless_columns(dataset: object) -> object:
    columns_to_drop = [
        col for col in dataset.column_names if col not in ["origin", "conversation"]
    ]
    dataset = dataset.remove_columns(columns_to_drop)
    return dataset


labeled_sets = [
    [detox_lmsys, "lmsys"],
    [detox_wildchat, "wildchat"],
    [detox_wildguard, "wildguard"],
    [detox_wildjb, "wildjailbreak"],
]

for i, entry in enumerate(labeled_sets):
    dataset, label = entry
    dataset = dataset.map(
        lambda x, label=label: label_dataset_sample(x, label), num_proc=NUM_CPUS
    )
    labeled_sets[i][0] = remove_useless_columns(dataset)

In [19]:
from datasets import concatenate_datasets

main_dataset = concatenate_datasets([d[0] for d in labeled_sets])

In [20]:
main_dataset

Dataset({
    features: ['conversation', 'origin'],
    num_rows: 1615349
})

In [ ]:
num_dataset_tokens = get_n_tokens(dataset=main_dataset, tokenizer=llama_tok)

Map (num_proc=8):   0%|          | 0/1615349 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (201581 > 131072). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (219196 > 131072). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (737471 > 131072). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (192102 > 131072). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (188811 > 131072). Running this sequence through the model will result in i

In [ ]:
from collections import Counter

from huggingface_hub import HfApi

REPO_ID = "ddidacus/guard-glp-benign"

composition = Counter(main_dataset["origin"])
comp_table = "\n".join(f"| {name} | {count:,} |" for name, count in composition.items())

card = f"""---
license: mit
---
# Guard-GLP Benign Conversations

Sanitized collection of benign multi-turn conversations, using **train splits only**.

## Composition
| Source | Samples |
|--------|---------|
{comp_table}
| **Total** | **{len(main_dataset):,}** |

**Total tokens:** {num_dataset_tokens:,} (Llama 3.2 1B Instruct tokenizer)

## Sanitization
- **LMSYS Chat 1M & WildChat**: filtered via OpenAI moderation labels (all flagged categories removed)
- **WildGuardMix**: removed adversarial and harmful-labeled prompts/responses
- **WildJailbreak**: removed entries with harmful data types

Only benign conversations are retained.
"""

main_dataset.push_to_hub(REPO_ID, private=True)

HfApi().upload_file(
    path_or_fileobj=card.encode(),
    path_in_repo="README.md",
    repo_id=REPO_ID,
    repo_type="dataset",
)
f"Pushed to {REPO_ID} | Total tokens: {num_dataset_tokens:,}"